<div style="background-color: #04D7FD; padding: 20px; text-align: left;">
    <h1 style="color: #000000; font-size: 36px; margin: 0;">Data Processing for RAG with Data Prep Kit (Python)</h1>
    
</div>


## Before Running the notebook

Please complete [setting up python dev environment](./setup-python-dev-env.md)

## Overview

This notebook will process PDF documents as part of RAG pipeline

![](media/rag-overview-2.png)

This notebook will perform steps 1, 2, 3 and 4 in RAG pipeline.

Here are the processing steps:

- **pdf2parquet** : Extract text (in markdown format) from PDF and store them as parquet files
- **Exact Dedup**: Documents with exact content are filtered out
- **Chunk documents**: Split the PDFs into 'meaningful sections' (paragraphs, sentences ..etc)
- **Text encoder**: Convert chunks into vectors using embedding models

## Step-1: Configuration

In [1]:
from my_config import MY_CONFIG

In [2]:
## setup path to utils folder
import sys
sys.path.append('../utils')

## Step-2:  Data

We will use white papers  about LLMs.  

- [Granite Code Models](https://arxiv.org/abs/2405.04324)
- [Attention is all you need](https://arxiv.org/abs/1706.03762)

You can of course substite your own data below

### 2.1 - Download data

In [3]:
import os, sys
import shutil
from file_utils import download_file

shutil.rmtree(MY_CONFIG.INPUT_DATA_DIR, ignore_errors=True)
shutil.os.makedirs(MY_CONFIG.INPUT_DATA_DIR, exist_ok=True)
print ("✅ Cleared input directory")
 
download_file (url = 'https://arxiv.org/pdf/1706.03762', local_file = os.path.join(MY_CONFIG.INPUT_DATA_DIR, 'attention.pdf' ))
download_file (url = 'https://arxiv.org/pdf/2405.04324', local_file = os.path.join(MY_CONFIG.INPUT_DATA_DIR, 'granite.pdf' ))
download_file (url = 'https://arxiv.org/pdf/2405.04324', local_file = os.path.join(MY_CONFIG.INPUT_DATA_DIR, 'granite2.pdf' )) # duplicate


✅ Cleared input directory

input/attention.pdf (2.22 MB) downloaded successfully.

input/granite.pdf (1.27 MB) downloaded successfully.

input/granite2.pdf (1.27 MB) downloaded successfully.


### 2.2 - Set input/output path variables for the pipeline

In [4]:
import os, sys
import shutil

if not os.path.exists(MY_CONFIG.INPUT_DATA_DIR ):
    raise Exception (f"❌ Input folder MY_CONFIG.INPUT_DATA_DIR = '{MY_CONFIG.INPUT_DATA_DIR}' not found")

output_parquet_dir = os.path.join (MY_CONFIG.OUTPUT_FOLDER, '01_parquet_out')
output_exact_dedupe_dir = os.path.join (MY_CONFIG.OUTPUT_FOLDER, '02_dedupe_out')
output_chunk_dir = os.path.join (MY_CONFIG.OUTPUT_FOLDER, '03_chunk_out')
output_embeddings_dir = os.path.join (MY_CONFIG.OUTPUT_FOLDER, '04_embeddings_out')

## clear output folder
shutil.rmtree(MY_CONFIG.OUTPUT_FOLDER, ignore_errors=True)
shutil.os.makedirs(MY_CONFIG.OUTPUT_FOLDER, exist_ok=True)

print ("✅ Cleared output directory")

✅ Cleared output directory


## Step-3: pdf2parquet -  Convert data from PDF to Parquet

This step is reading the input folder containing all PDF files and ingest them in a parquet table using the [Docling package](https://github.com/DS4SD/docling).
The documents are converted into a JSON format which allows to easily chunk it in the later steps.



### 3.1 - Execute 

In [5]:
%%time 

from dpk_pdf2parquet.transform_python import Pdf2Parquet
from dpk_pdf2parquet.transform import pdf2parquet_contents_types

STAGE = 1 
print (f"🏃🏼 STAGE-{STAGE}: Processing input='{MY_CONFIG.INPUT_DATA_DIR}' --> output='{output_parquet_dir}'\n", flush=True)

result = Pdf2Parquet(input_folder= MY_CONFIG.INPUT_DATA_DIR,
                    output_folder= output_parquet_dir,
                    data_files_to_use=['.pdf'],
                    pdf2parquet_contents_type=pdf2parquet_contents_types.MARKDOWN,   # markdown
                    #    pdf2parquet_contents_type=pdf2parquet_contents_types.JSON   # JSON
                    ).transform()

if result == 0:
    print (f"✅ Stage:{STAGE} completed successfully")
else:
    raise Exception (f"❌ Stage:{STAGE}  failed")

🏃🏼 STAGE-1: Processing input='input' --> output='output/01_parquet_out'



00:42:17 INFO - pdf2parquet parameters are : {'batch_size': -1, 'artifacts_path': None, 'contents_type': <pdf2parquet_contents_types.MARKDOWN: 'text/markdown'>, 'do_table_structure': True, 'do_ocr': True, 'ocr_engine': <pdf2parquet_ocr_engine.EASYOCR: 'easyocr'>, 'bitmap_area_threshold': 0.05, 'pdf_backend': <pdf2parquet_pdf_backend.DLPARSE_V2: 'dlparse_v2'>, 'double_precision': 8}
00:42:17 INFO - pipeline id pipeline_id
00:42:17 INFO - code location None
00:42:17 INFO - data factory data_ is using local data access: input_folder - input output_folder - output/01_parquet_out
00:42:17 INFO - data factory data_ max_files -1, n_sample -1
00:42:17 INFO - data factory data_ Not using data sets, checkpointing False, max files -1, random samples -1, files to use ['.pdf'], files to checkpoint ['.parquet']
00:42:17 INFO - orchestrator pdf2parquet started at 2025-03-14 00:42:17
00:42:17 INFO - Number of files is 3, source profile {'max_file_size': 2.112621307373047, 'min_file_size': 1.2146415710

✅ Stage:1 completed successfully
CPU times: user 1min 2s, sys: 2.06 s, total: 1min 4s
Wall time: 58.9 s


### 3.2 -  Inspect Generated output

Here we should see one entry per input file processed

In [6]:
from file_utils import read_parquet_files_as_df

output_df = read_parquet_files_as_df(output_parquet_dir)

# print ("Output dimensions (rows x columns)= ", output_df.shape)

output_df.head(5)

## To display certain columns
#parquet_df[['column1', 'column2', 'column3']].head(5)

Successfully read 3 parquet files with 3 total rows


,filename,contents,num_pages,num_tables,num_doc_elements,document_id,document_hash,ext,hash,size,date_acquired,pdf_convert_time,source_filename
0,attention.pdf,"Provided proper attribution is provided, Googl...",15,6,436,99d94af1-adc6-459c-b1ed-a1d7e3449ab5,2949302674760005271,pdf,5bbff6b2beac5f09f368a913932212e8d57953b3e3e86d...,45855,2025-03-14T00:42:33.881031,12.903253,attention.pdf
1,granite2.pdf,## Granite Code Models: A Family of Open Found...,28,21,324,d0bd330b-e2d1-460c-8b6f-30b3f7363a70,3127757990743433032,pdf,bf9642b9f719c6c7db1ad66a2992c96a755339ba9e793e...,136086,2025-03-14T00:43:11.972510,18.821723,granite2.pdf
2,granite.pdf,## Granite Code Models: A Family of Open Found...,28,21,324,7762f80d-e103-41e5-bce0-4688219634db,3127757990743433032,pdf,bf9642b9f719c6c7db1ad66a2992c96a755339ba9e793e...,136086,2025-03-14T00:42:53.104790,19.178672,granite.pdf


## Step-4: Eliminate Duplicate Documents

We have 2 duplicate documnets here : `granite.pdf` and `granite2.pdf`.

Note how the `hash` for these documents are same.

We are going to perform **de-dupe**

On the content of each document, a SHA256 hash is computed, followed by de-duplication of record having identical hashes.

[Dedupe transform documentation](https://github.com/IBM/data-prep-kit/blob/dev/transforms/universal/ededup/README.md)

### 4.1 - Execute 

In [7]:
%%time 

from dpk_ededup.transform_python import Ededup

STAGE = 2
print (f"🏃🏼 STAGE-{STAGE}: Processing input='{output_parquet_dir}' --> output='{output_exact_dedupe_dir}'\n", flush=True)

result = Ededup(input_folder=output_parquet_dir,
    output_folder=output_exact_dedupe_dir,
    ededup_doc_column="contents",
    ededup_doc_id_column="document_id"
    ).transform()

if result == 0:
    print (f"✅ Stage:{STAGE} completed successfully")
else:
    raise Exception (f"❌ Stage:{STAGE}  failed")

🏃🏼 STAGE-2: Processing input='output/01_parquet_out' --> output='output/02_dedupe_out'



00:43:12 INFO - exact dedup params are {'doc_column': 'contents', 'doc_id_column': 'document_id', 'use_snapshot': False, 'snapshot_directory': None}
00:43:12 INFO - pipeline id pipeline_id
00:43:12 INFO - code location None
00:43:12 INFO - data factory data_ is using local data access: input_folder - output/01_parquet_out output_folder - output/02_dedupe_out
00:43:12 INFO - data factory data_ max_files -1, n_sample -1
00:43:12 INFO - data factory data_ Not using data sets, checkpointing False, max files -1, random samples -1, files to use ['.parquet'], files to checkpoint ['.parquet']
00:43:12 INFO - orchestrator ededup started at 2025-03-14 00:43:12
00:43:12 INFO - Number of files is 3, source profile {'max_file_size': 0.0434112548828125, 'min_file_size': 0.020546913146972656, 'total_file_size': 0.10735607147216797}
00:43:12 INFO - Starting from the beginning
00:43:12 INFO - Completed 1 files (33.33%) in 0.0 min
00:43:12 INFO - Completed 2 files (66.67%) in 0.0 min
00:43:12 INFO - Com

✅ Stage:2 completed successfully
CPU times: user 47.1 ms, sys: 2.09 ms, total: 49.1 ms
Wall time: 44.4 ms


### 4.2 - Inspect Generated output

We would see 2 documents: `attention.pdf`  and `granite.pdf`.  The duplicate `granite.pdf` has been filtered out!

In [8]:
from file_utils import read_parquet_files_as_df

input_df = read_parquet_files_as_df(output_parquet_dir)
output_df = read_parquet_files_as_df(output_exact_dedupe_dir)

# print ("Input data dimensions (rows x columns)= ", input_df.shape)
# print ("Output data dimensions (rows x columns)= ", output_df.shape)
print (f"Input files before exact dedupe : {input_df.shape[0]:,}")
print (f"Output files after exact dedupe : {output_df.shape[0]:,}")
print ("Duplicate files removed :  ", (input_df.shape[0] - output_df.shape[0]))

output_df.sample(min(3, output_df.shape[0]))

Successfully read 3 parquet files with 3 total rows
Successfully read 2 parquet files with 2 total rows
Input files before exact dedupe : 3
Output files after exact dedupe : 2
Duplicate files removed :   1


,filename,contents,num_pages,num_tables,num_doc_elements,document_id,document_hash,ext,hash,size,date_acquired,pdf_convert_time,source_filename,removed
0,attention.pdf,"Provided proper attribution is provided, Googl...",15,6,436,99d94af1-adc6-459c-b1ed-a1d7e3449ab5,2949302674760005271,pdf,5bbff6b2beac5f09f368a913932212e8d57953b3e3e86d...,45855,2025-03-14T00:42:33.881031,12.903253,attention.pdf,[]
1,granite.pdf,## Granite Code Models: A Family of Open Found...,28,21,324,7762f80d-e103-41e5-bce0-4688219634db,3127757990743433032,pdf,bf9642b9f719c6c7db1ad66a2992c96a755339ba9e793e...,136086,2025-03-14T00:42:53.104790,19.178672,granite.pdf,[]


##  Step-5: Doc chunks

Split the documents in chunks.

[Chunking transform documentation](https://github.com/IBM/data-prep-kit/blob/dev/transforms/language/doc_chunk/README.md)

**Experiment with chunking size to find the setting that works best for your documents**

### 5.1 - Execute 

In [9]:
%%time

from dpk_doc_chunk.transform_python import DocChunk

STAGE = 3
print (f"🏃🏼 STAGE-{STAGE}: Processing input='{output_exact_dedupe_dir}' --> output='{output_chunk_dir}'\n", flush=True)

result = DocChunk(input_folder=output_exact_dedupe_dir,
        output_folder=output_chunk_dir,
        doc_chunk_chunking_type= "li_markdown",
        # doc_chunk_chunking_type= "dl_json",
        doc_chunk_chunk_size_tokens = 128,  # default 128
        doc_chunk_chunk_overlap_tokens=30   # default 30
        ).transform()

if result == 0:
    print (f"✅ Stage:{STAGE} completed successfully")
else:
    raise Exception (f"❌ Stage:{STAGE}  failed")

🏃🏼 STAGE-3: Processing input='output/02_dedupe_out' --> output='output/03_chunk_out'



00:43:12 INFO - doc_chunk parameters are : {'chunking_type': 'li_markdown', 'content_column_name': 'contents', 'doc_id_column_name': 'document_id', 'output_chunk_column_name': 'contents', 'output_source_doc_id_column_name': 'source_document_id', 'output_jsonpath_column_name': 'doc_jsonpath', 'output_pageno_column_name': 'page_number', 'output_bbox_column_name': 'bbox', 'chunk_size_tokens': 128, 'chunk_overlap_tokens': 30, 'dl_min_chunk_len': None}
00:43:12 INFO - pipeline id pipeline_id
00:43:12 INFO - code location None
00:43:12 INFO - data factory data_ is using local data access: input_folder - output/02_dedupe_out output_folder - output/03_chunk_out
00:43:12 INFO - data factory data_ max_files -1, n_sample -1
00:43:12 INFO - data factory data_ Not using data sets, checkpointing False, max files -1, random samples -1, files to use ['.parquet'], files to checkpoint ['.parquet']
00:43:12 INFO - orchestrator doc_chunk started at 2025-03-14 00:43:12
00:43:12 INFO - Number of files is 3,

✅ Stage:3 completed successfully
CPU times: user 673 ms, sys: 98.8 ms, total: 772 ms
Wall time: 779 ms


### 5.2 - Inspect Generated output

We would see documents are split into many chunks

In [10]:
from file_utils import read_parquet_files_as_df

input_df = read_parquet_files_as_df(output_exact_dedupe_dir)  ## for debug purposes
output_df = read_parquet_files_as_df(output_chunk_dir)

print (f"Files processed : {input_df.shape[0]:,}")
print (f"Chunks created : {output_df.shape[0]:,}")

# print ("Input data dimensions (rows x columns)= ", input_df.shape)
# print ("Output data dimensions (rows x columns)= ", output_df.shape)

output_df.sample(min(3, output_df.shape[0]))

Successfully read 2 parquet files with 2 total rows
Successfully read 2 parquet files with 61 total rows
Files processed : 2
Chunks created : 61


,filename,num_pages,num_tables,num_doc_elements,document_hash,ext,hash,size,date_acquired,pdf_convert_time,source_filename,removed,source_document_id,contents,document_id
23,attention.pdf,15,6,436,2949302674760005271,pdf,5bbff6b2beac5f09f368a913932212e8d57953b3e3e86d...,45855,2025-03-14T00:42:33.881031,12.903253,attention.pdf,[],99d94af1-adc6-459c-b1ed-a1d7e3449ab5,## 6.2 Model Variations\n\nTo evaluate the imp...,dd05d18118cebfeb1c6eafee2faa66e0016c5966d29209...
29,granite.pdf,28,21,324,3127757990743433032,pdf,bf9642b9f719c6c7db1ad66a2992c96a755339ba9e793e...,136086,2025-03-14T00:42:53.104790,19.178672,granite.pdf,[],7762f80d-e103-41e5-bce0-4688219634db,## Abstract\n\nLarge Language Models (LLMs) tr...,fa843bc8b05ba80dd1ee780200b1614af27a2818759fa4...
32,granite.pdf,28,21,324,3127757990743433032,pdf,bf9642b9f719c6c7db1ad66a2992c96a755339ba9e793e...,136086,2025-03-14T00:42:53.104790,19.178672,granite.pdf,[],7762f80d-e103-41e5-bce0-4688219634db,## 2.1 Data Crawling and Filtering\n\nThe pret...,64f536c6e279db81ccda1ba14e35ca553e4250de346176...


## Step-6:   Calculate Embeddings for Chunks

we will calculate embeddings for each chunk using an open source embedding model

[Embeddings / Text Encoder documentation](https://github.com/IBM/data-prep-kit/blob/dev/transforms/language/text_encoder/README.md)

### 6.1 - Execute

In [11]:
%%time 

from dpk_text_encoder.transform_python import TextEncoder

STAGE  = 4
print (f"🏃🏼 STAGE-{STAGE}: Processing input='{output_chunk_dir}' --> output='{output_embeddings_dir}'\n", flush=True)


result = TextEncoder(input_folder= output_chunk_dir, 
               output_folder= output_embeddings_dir, 
               text_encoder_model_name = MY_CONFIG.EMBEDDING_MODEL
               ).transform()
if result == 0:
    print (f"✅ Stage:{STAGE} completed successfully")
else:
    raise Exception (f"❌ Stage:{STAGE}  failed")

🏃🏼 STAGE-4: Processing input='output/03_chunk_out' --> output='output/04_embeddings_out'



00:43:13 INFO - text_encoder parameters are : {'content_column_name': 'contents', 'output_embeddings_column_name': 'embeddings', 'model_name': 'ibm-granite/granite-embedding-30m-english'}
00:43:13 INFO - pipeline id pipeline_id
00:43:13 INFO - code location None
00:43:13 INFO - data factory data_ is using local data access: input_folder - output/03_chunk_out output_folder - output/04_embeddings_out
00:43:13 INFO - data factory data_ max_files -1, n_sample -1
00:43:13 INFO - data factory data_ Not using data sets, checkpointing False, max files -1, random samples -1, files to use ['.parquet'], files to checkpoint ['.parquet']
00:43:13 INFO - orchestrator text_encoder started at 2025-03-14 00:43:13
00:43:13 INFO - Number of files is 2, source profile {'max_file_size': 0.04574298858642578, 'min_file_size': 0.028684616088867188, 'total_file_size': 0.07442760467529297}
00:43:15 INFO - Completed 1 files (50.0%) in 0.003 min
00:43:15 INFO - Completed 2 files (100.0%) in 0.007 min
00:43:15 INF

✅ Stage:4 completed successfully
CPU times: user 987 ms, sys: 167 ms, total: 1.15 s
Wall time: 2.5 s


### 6.2 - Inspect Generated output

In [12]:
from file_utils import read_parquet_files_as_df

input_df = read_parquet_files_as_df(output_chunk_dir)
output_df = read_parquet_files_as_df(output_embeddings_dir)

print ("Input data dimensions (rows x columns)= ", input_df.shape)
print ("Output data dimensions (rows x columns)= ", output_df.shape)

output_df.sample(min(3, output_df.shape[0]))

Successfully read 2 parquet files with 61 total rows
Successfully read 2 parquet files with 61 total rows
Input data dimensions (rows x columns)=  (61, 15)
Output data dimensions (rows x columns)=  (61, 16)


,filename,num_pages,num_tables,num_doc_elements,document_hash,ext,hash,size,date_acquired,pdf_convert_time,source_filename,removed,source_document_id,contents,document_id,embeddings
17,attention.pdf,15,6,436,2949302674760005271,pdf,5bbff6b2beac5f09f368a913932212e8d57953b3e3e86d...,45855,2025-03-14T00:42:33.881031,12.903253,attention.pdf,[],99d94af1-adc6-459c-b1ed-a1d7e3449ab5,## 5.1 Training Data and Batching\n\nWe traine...,ded755bb92c47f5736c352e9cce8db2c54023775c202d9...,"[0.051321298, -0.023440624, 0.08599294, -0.008..."
55,granite.pdf,28,21,324,3127757990743433032,pdf,bf9642b9f719c6c7db1ad66a2992c96a755339ba9e793e...,136086,2025-03-14T00:42:53.104790,19.178672,granite.pdf,[],7762f80d-e103-41e5-bce0-4688219634db,## 6.6 Calling Functions and Tools\n\nWe adopt...,7eef602c19b4a0b1926d76fbf321223285438562e9e392...,"[-0.053145144, 0.036063302, 0.022222947, -0.03..."
42,granite.pdf,28,21,324,3127757990743433032,pdf,bf9642b9f719c6c7db1ad66a2992c96a755339ba9e793e...,136086,2025-03-14T00:42:53.104790,19.178672,granite.pdf,[],7762f80d-e103-41e5-bce0-4688219634db,## 5 Instruction Tuning\n\nFinetuning code LLM...,90cee92e6eb1491c33f3763f77d777a5c7a5329a17aa00...,"[0.03946444, -0.031164119, 0.026932104, 0.0314..."


## Step-7: Copy output to final output dir

In [13]:
import shutil

shutil.rmtree(MY_CONFIG.OUTPUT_FOLDER_FINAL, ignore_errors=True)
shutil.copytree(src=output_embeddings_dir, dst=MY_CONFIG.OUTPUT_FOLDER_FINAL)

print (f"✅ Copied output from '{output_embeddings_dir}' --> '{MY_CONFIG.OUTPUT_FOLDER_FINAL}'")

✅ Copied output from 'output/04_embeddings_out' --> 'output/output_final'
